# Keypoint moseq data post-processing notebook

### Plotting animal's ethograms

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def contiguous_segments(states):
    """
    Given an array of states, return a list of tuples (start, end, state)
    for contiguous segments. Here, start and end are indices (inclusive).
    """
    segments = []
    start = 0
    current_state = states[0]
    for i in range(1, len(states)):
        if states[i] != current_state:
            segments.append((start, i - 1, current_state))
            start = i
            current_state = states[i]
    segments.append((start, len(states) - 1, current_state))
    return segments

def plot_all_together(states, ax):
    """
    Plot a single-row ethogram with colored spans for each contiguous segment.
    Also, add a legend with alphabetically sorted state labels.
    """
    segments = contiguous_segments(states)
    # Get sorted unique states to fix the legend order
    unique_states = sorted(np.unique(states))
    cmap = plt.get_cmap("Set2")
    color_dict = {state: cmap(i % 10) for i, state in enumerate(unique_states)}
    
    # Plot each segment as a colored span along one horizontal line
    for seg in segments:
        start, end, state = seg
        # end+1 to account for inclusive end index (adjust if desired)
        ax.axvspan(start, end + 1, color=color_dict[state], alpha=0.7, label=str(state))
    
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time frame")
    ax.set_yticks([])
    
    # Build a legend with sorted labels
    handles, labels = ax.get_legend_handles_labels()
    legend_dict = {}
    for h, lab in zip(handles, labels):
        if lab not in legend_dict:
            legend_dict[lab] = h
    sorted_labels = sorted(legend_dict.keys())
    sorted_handles = [legend_dict[lab] for lab in sorted_labels]
    ax.legend(sorted_handles, sorted_labels, title="Syllable States", loc='upper right')
    
def plot_split(states, ax):
    """
    Plot an ethogram where each unique syllable state is assigned its own row.
    Contiguous segments for a state are drawn on that row.
    """
    segments = contiguous_segments(states)
    unique_states = sorted(np.unique(states))
    cmap = plt.get_cmap("Set2")
    color_dict = {state: cmap(i % 10) for i, state in enumerate(unique_states)}
    
    # Map each state to a row index
    state_to_row = {state: i for i, state in enumerate(unique_states)}
    
    for seg in segments:
        start, end, state = seg
        row = state_to_row[state]
        # Use broken_barh: note that width is (end - start + 1)
        ax.broken_barh([(start, end - start + 1)], (row - 0.4, 0.8), facecolors=color_dict[state])
    
    ax.set_xlabel("Time frame")
    ax.set_yticks(range(len(unique_states)))
    ax.set_yticklabels(unique_states)
    ax.set_ylim(-1, len(unique_states))
    
    # Build legend manually using custom line objects with a smaller font size
    from matplotlib.lines import Line2D
    legend_handles = [Line2D([0], [0], color=color_dict[state], lw=4) for state in unique_states]
    ax.legend(legend_handles, unique_states, title="Syllable States", loc='upper right', prop={'size': 8})
    
def process_csv_file(csv_path):
    """
    Given a CSV file path, create two ethogram plots:
      1. All syllables together (one horizontal timeline)
      2. Each syllable on its own row.
    Save the plots in the same directory as the CSV file using the same base name.
    """
    df = pd.read_csv(csv_path)
    if 'syllable' not in df.columns:
        print(f"File {csv_path} does not contain a 'syllable' column. Skipping.")
        return
    states = df['syllable'].values
    base, _ = os.path.splitext(csv_path)
    
    # Plot: all syllables together
    fig, ax = plt.subplots(figsize=(15, 2))
    plot_all_together(states, ax)
    ax.set_title("All syllables together")
    all_output_path = base + "_ethogram_all.png"
    fig.savefig(all_output_path)
    plt.close(fig)
    
    # Plot: each syllable on a separate row
    fig, ax = plt.subplots(figsize=(15, 2))
    plot_split(states, ax)
    ax.set_title("Each syllable on separate row")
    split_output_path = base + "_ethogram_split.png"
    fig.savefig(split_output_path)
    plt.close(fig)
    
    print(f"Saved plots for {csv_path} as:\n  {all_output_path}\n  {split_output_path}")

def process_directory(directory):
    """
    For each CSV file in the specified directory, process the file.
    """
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            csv_path = os.path.join(directory, filename)
            process_csv_file(csv_path)

# Example usage:
directory = '/Volumes/ANNA_HD/ANALYSIS/EXPERIMENTS/2024/3cham_wanhui/kpms/feb25/kpms-3cham/2025_02_27-23_25_06/results'
process_directory(directory)


### Syllable heatmaps

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from matplotlib.patches import Rectangle, Circle

def load_roi_data(roi_file):
    """
    Load the ROI data from the HDF5 file (under 'coordinates').
    This function assumes the file contains datasets with roi information (e.g., coordinates, size, etc.).
    """
    with h5py.File(roi_file, 'r') as f:
        coordinates_group = f['coordinates']
        
        # Extract the relevant data from block0_values (ROI coordinates)
        block0_values = coordinates_group['block0_values'][:]
        
        # Extract the shape-type information from block2_values
        shape_type = coordinates_group['block2_values'][0]  # Assuming the first element contains shape type info
        
        print(f"Shape type: {shape_type}")  # Debugging line to inspect the shape type
        
        # Prepare the ROI data from coordinates and shape type information
        rois = []
        for roi in block0_values:
            roi_data = {
                'x': roi[0],  # x-coordinate
                'y': roi[1],  # y-coordinate
                'type': str(shape_type[0]) if isinstance(shape_type[0], bytes) else shape_type[0],  # Handle byte string or numeric types
            }
            
            # Here, assume shape info is based on `shape_type` values (you can expand based on actual dataset structure)
            if 'width' in roi and 'height' in roi:
                roi_data['width'] = roi[2]  # Width (for rectangles)
                roi_data['height'] = roi[3]  # Height (for rectangles)
            elif 'radius' in roi:
                roi_data['radius'] = roi[2]  # Radius (for circles)
            
            rois.append(roi_data)
    
    return rois

def plot_syllable_kde_with_rois(df, syllable, ax, bins=50, range_x=(0, 500), range_y=(0, 500), roi_file=None):
    """
    Plot a KDE plot for centroid-x and centroid-y coordinates of a specific syllable.
    Also, plot ROIs as rectangular and circular patches on top of the KDE.
    """
    # Filter the dataframe for the given syllable
    syllable_df = df[df['syllable'] == syllable]
    
    # Check if there is enough data for this syllable
    if len(syllable_df) == 0:
        print(f"⚠️ No data for syllable {syllable}. Skipping KDE plot.")
        return
    
    # Extract centroid x and y coordinates
    x_coords = syllable_df['centroid x'].values
    y_coords = syllable_df['centroid y'].values
    
    # Skip if the data has 0 variance
    if np.var(x_coords) == 0 or np.var(y_coords) == 0:
        print(f"⚠️ No variance in centroid coordinates for syllable {syllable}. Skipping KDE plot.")
        return
    
    # Create the KDE plot
    sns.kdeplot(x=x_coords, y=y_coords, ax=ax, cmap="Blues", shade=True, bw_adjust=0.5, fill=True)
    
    # Set the title and labels
    ax.set_title(f"Syllable {syllable} KDE Plot", fontsize=12)
    ax.set_xlabel("Centroid X", fontsize=10)
    ax.set_ylabel("Centroid Y", fontsize=10)
    
    # Preserve the aspect ratio to match the arena's rectangular shape
    ax.set_aspect('auto', adjustable='box')
    
    # If ROI file is provided, plot the ROIs
    if roi_file:
        rois = load_roi_data(roi_file)
        for roi in rois:
            x = roi['x']
            y = roi['y']
            if 'width' in roi and 'height' in roi:
                # Rectangle ROI
                rect = Rectangle((x, y), roi['width'], roi['height'], linewidth=1, edgecolor='r', facecolor='none')
                ax.add_patch(rect)
            elif 'radius' in roi:
                # Circle ROI
                circle = Circle((x, y), roi['radius'], linewidth=1, edgecolor='g', facecolor='none')
                ax.add_patch(circle)
    
    # Add a colorbar
    if len(ax.collections) > 0:
        cbar = plt.colorbar(ax.collections[0], ax=ax)
        cbar.set_label('Density', fontsize=10)

def process_csv_file_with_rois(csv_path, roi_dir):
    """
    Process the CSV file and plot the KDE plot with ROIs.
    """
    df = pd.read_csv(csv_path)
    
    if 'syllable' not in df.columns or 'centroid x' not in df.columns or 'centroid y' not in df.columns:
        print(f"File {csv_path} does not contain necessary columns ('syllable', 'centroid_x', 'centroid_y'). Skipping.")
        return
    
    # Filter syllables 0-8 with min_freq=0.001
    min_freq = 0.001
    df_filtered = df[df['syllable'].isin(range(9))]  # Only syllables 0-8
    
    # Get the base name of the file and create a directory for this animal
    base, _ = os.path.splitext(csv_path)
    animal_dir = base + "_kde_plots"
    os.makedirs(animal_dir, exist_ok=True)
    
    # For each syllable (0-8), plot and save the KDE plot with ROIs
    unique_syllables = df_filtered['syllable'].unique()
    for syllable in unique_syllables:
        fig, ax = plt.subplots(figsize=(6, 6))
        
        # Find the corresponding ROI file by stripping everything after 'DLC_' in the CSV filename
        roi_filename = base.split('/')[-1].split('DLC_')[0] + "_roi.h5"  # Correct the filename structure
        roi_file = os.path.join(roi_dir, roi_filename)
        
        # Plot the KDE with ROIs
        plot_syllable_kde_with_rois(df_filtered, syllable, ax, roi_file=roi_file)
        
        kde_path = os.path.join(animal_dir, f"syllable_{syllable}_kde_with_rois.png")
        plt.savefig(kde_path)
        plt.close(fig)
        print(f"Saved KDE plot with ROIs for syllable {syllable} in {kde_path}")
    
def process_directory(directory, roi_dir):
    """
    Process each CSV file in the given directory and generate syllable KDE plots with ROIs.
    """
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            csv_path = os.path.join(directory, filename)
            process_csv_file_with_rois(csv_path, roi_dir)


directory = '/Users/annateruel/Desktop/results'
roi_directory = '/Users/annateruel/Desktop/all_dlc'
process_directory(directory, roi_directory)

In [ ]:
import cv2
import os
import numpy as np
import pandas as pd
from abc import ABC, abstractmethod
from qtpy.QtWidgets import QPushButton, QVBoxLayout, QWidget
from shapely.geometry import Point, Polygon, box

class ROI(ABC):
    """
    Abstract base class for a region of interest (ROI).
    """  
    @abstractmethod
    def is_point_inside_roi(self, point):
        """
        Determines if a given point is inside the ROI.

        Args:
            point (tuple): A tuple representing the (x, y) coordinates of the point to check.

        Returns:
            bool: True if the point is inside the ROI, False otherwise.
        """       
        pass
    
class EllipseROI(ROI):
    """
    Ellipse region of interest class
    """   
    def __init__(self, center, rad):
        """
        Initializing the ellipse roi

        Attributes:
            center (tuple): The (x, y) coordinates of the ellipse's center.
            rad (tuple): A tuple representing the semi-major and semi-minor radii of the ellipse.
        """         
        self.center = center
        self.rad = rad

    @classmethod
    def extract_ellipses(cls, df):
        """
        Class method to extract ellipses from a DataFrame (output from ROIDrawer)

        Args:
            df (pd.DataFrame): DataFrame containing ellipse data.

        Returns:
            tuple: A tuple containing a list of EllipseROI objects and a dictionary of their parameters.
        """       
        def ellipse_callback(group):
            axis_0_values = group['axis-0'].values
            axis_1_values = group['axis-1'].values
            center = ((axis_0_values.min() + axis_0_values.max()) / 2, # x-coordinate
                      (axis_1_values.min() + axis_1_values.max()) / 2) # y-coordinate
            radii = ((axis_0_values.max() - axis_0_values.min()) / 2, # semi MINOR axis
                     (axis_1_values.max() - axis_1_values.min()) / 2) #s emi MAJOR axis
            return {'roi': cls(center, radii), 'params': {'center': center, 'rad': radii}}
        return extract_roi_data(df, 'add_ellipse', ellipse_callback)

    def is_point_inside_roi(self, point):
        """
        Determines if a given point is inside the ellipse.

        Args:
            point (tuple): A tuple representing the (x, y) coordinates of the point to check.

        Returns:
            bool: True if the point is inside the ellipse, False otherwise.
        """        
        x, y = point
        h, k = self.center
        a, b = self.rad
        return ((x - h)**2 / a**2) + ((y - k)**2 / b**2) <= 1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import pandas as pd
import numpy as np
import h5py  # Add this line at the top of your script


def load_roi_ellipse(roi_file_path):
    """
    Load ROI data from an HDF5 file and return a list of EllipseROI objects.

    Args:
        roi_file_path (str): Path to the .h5 file containing the ROI data.

    Returns:
        List[EllipseROI]: A list of EllipseROI objects extracted from the file.
    """
    roi_list = []
    try:
        with h5py.File(roi_file_path, 'r') as f:
            coordinates_group = f['coordinates']
            
            # Check for the 'axis-0' and 'axis-1' data
            axis_0 = coordinates_group['axis0'][:]
            axis_1 = coordinates_group['axis1'][:]
            
            # Extract center and radii for each ROI
            center_x = (axis_0.min() + axis_0.max()) / 2
            center_y = (axis_1.min() + axis_1.max()) / 2
            radii_x = (axis_0.max() - axis_0.min()) / 2
            radii_y = (axis_1.max() - axis_1.min()) / 2
            
            # Create an EllipseROI object and append to the list
            roi = EllipseROI(center=(center_x, center_y), rad=(radii_x, radii_y))
            roi_list.append(roi)

    except Exception as e:
        print(f"Error loading ROI from {roi_file_path}: {e}")
    
    return roi_list
def plot_kde_and_rois(roi_ellipse, video_df, syllable, ax):
    """Plot KDE for centroids and overlay ellipses for ROIs."""
    
    # Get the video data for the given syllable (centroid positions)
    syllable_data = video_df[video_df['syllable'] == syllable]
    x_coords = syllable_data['centroid_x'].values
    y_coords = syllable_data['centroid_y'].values
    
    # Plot KDE for centroid distribution of the syllable
    sns.kdeplot(x=x_coords, y=y_coords, cmap='Blues', fill=True, ax=ax)

    # Overlay ellipses (ROIs) for each animal in the roi_ellipse dictionary
    for animal_name, rois in roi_ellipse.items():
        for roi in rois:
            center_x, center_y = roi.center
            radius_x, radius_y = roi.rad

            # Create the ellipse and add it to the plot
            ellipse = Ellipse(
                (center_x, center_y),
                width=2 * radius_x,  # Multiply radii by 2 to get full width/height
                height=2 * radius_y,
                edgecolor='red',
                facecolor='none',
                lw=2
            )
            ax.add_patch(ellipse)
            ax.legend()

    ax.set_xlabel("Centroid X")
    ax.set_ylabel("Centroid Y")
    ax.set_title(f"Syllable {syllable} with ROIs")

def process_video_for_kde(directory, roi_directory):
    """Process video and ROI files and plot KDE with ROIs for each syllable."""
    roi_ellipse = {}

    # Read the ROI files and extract ellipse data
    for file in os.listdir(roi_directory):
        if file.endswith('roi.h5'):
            roi_file_path = os.path.join(roi_directory, file)
            print(f"Processing ROI file: {file}")
            animal_name = os.path.basename(file).replace("_roi.h5", "")  # Extract animal name
            rois = load_roi_ellipse(roi_file_path)  # Assuming `load_roi_ellipse` returns EllipseROI objects
            roi_ellipse[animal_name] = rois

    # Process each video file and plot KDE with ROIs for each syllable
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            csv_path = os.path.join(directory, filename)
            df = pd.read_csv(csv_path)

            # Get the base name for the animal (remove DLC_ and _filtered.csv)
            animal_name = os.path.basename(csv_path).split('DLC_')[1].replace('_filtered.csv', '')

            # Create the plot
            fig, ax = plt.subplots(figsize=(10, 10))

            # Plot KDE and ROIs for all syllables (assuming syllables 0-8)
            for syllable in range(9):
                plot_kde_and_rois(roi_ellipse, df, syllable, ax)

            # Save the plot
            plot_path = os.path.join(directory, f"{animal_name}_kde_with_rois.png")
            plt.savefig(plot_path)
            plt.close(fig)
            print(f"Saved plot for {animal_name} at {plot_path}")

# Example usage
directory = '/Users/annateruel/Desktop/results'  # Directory with centroid CSV files
roi_directory = '/Users/annateruel/Desktop/all_dlc'  # Directory with ROI .h5 files
process_video_for_kde(directory, roi_directory)